# Full pipeline — from raw API data to every reported number
**Which AI-augmented papers actually gain impact?** — Tadesse Birara Akele (2026A8019427001) · Deonisio Yohanes Wilfridus Hornay (026A8000531004)

This notebook reruns the **entire frozen pipeline** end to end. Each stage is a plain script under `src/`, driven only by `configs/config.yaml` (all seeds = 42, frozen split, frozen lexicon).

| stage | script | time | output |
|---|---|---|---|
| 1. fetch | `src/fetch_data.py` | ~4 min (network) | `data/raw/epmc_raw.jsonl` |
| 2. build dataset | `src/build_dataset.py` | <1 min | `data/processed/dataset.csv`, `results/label_stats.json` |
| 3. EDA figures | `src/make_eda_figures.py` | <1 min | `results/figures/eda_*.png` |
| 4. train + evaluate | `src/train_evaluate.py` | a few min (200× bootstrap) | `results/metrics.json`, `test_predictions.csv`, figures |
| 5. association + uplift | `src/association_and_uplift.py` | a few min (50× bootstrap) | `results/metrics_supplement.json`, `metrics_uplift_boot.json`, `uplift.png` |

> **Note on stage 1:** the repository ships the processed dataset but not the raw JSONL (size). Stage 1 re-downloads it deterministically from Europe PMC — same seed, same windows, same records. If you have no network, skip stage 1 and the rest still runs against the shipped `dataset.csv`.

## 0. Setup

In [ ]:
from pathlib import Path
import subprocess, sys, time

ROOT = Path.cwd()
if not (ROOT / "configs/config.yaml").exists():
    ROOT = ROOT.parent          # launched from notebooks/
assert (ROOT / "configs/config.yaml").exists()

def run(stage, script):
    t0 = time.time()
    print(f"--- stage {stage}: {script} ---")
    r = subprocess.run([sys.executable, str(ROOT / script)], cwd=ROOT,
                       capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise SystemExit(f"stage {stage} failed")
    print(f"done in {time.time()-t0:.0f}s")

## 1. Fetch raw records from Europe PMC (deterministic, seed 42)
24 random 2-day windows per year 2015–2021, first 160 records per window by publication date. Requires network; ~4 minutes.

In [ ]:
# Skip automatically if a raw file already exists
if (ROOT / "data/raw/epmc_raw.jsonl").exists():
    print("raw file already present — skipping fetch")
else:
    run("1 (fetch)", "src/fetch_data.py")

## 2. Build the analysis dataset
Parses JSONL, applies the AI lexicon, builds within-year top-10/5/20 outcomes, computes journal prestige and the 40-keyword vocabulary from **train years only**, subsamples to 25,000 (seed 42).

In [ ]:
run("2 (build)", "src/build_dataset.py")

## 3. EDA figures

In [ ]:
run("3 (eda)", "src/make_eda_figures.py")

## 4. Train + evaluate (baseline vs. challenger, ablations, sensitivity, bootstrap CIs)

In [ ]:
run("4 (train)", "src/train_evaluate.py")

## 5. Association, interaction model, and subgroup uplift (50 bootstrap refits)

In [ ]:
run("5 (association)", "src/association_and_uplift.py")

## 6. Confirm the regenerated outputs match the shipped frozen ones
If you ran stages 1–5, the JSONs were just overwritten by identical values (same seeds, same config).

In [ ]:
import json, hashlib
for f in ["results/metrics.json", "results/metrics_supplement.json",
          "results/metrics_uplift_boot.json", "results/label_stats.json"]:
    h = hashlib.sha256((ROOT / f).read_bytes()).hexdigest()[:16]
    print(f"{f:42s} sha256:{h}...")
m = json.load(open(ROOT / "results/metrics.json"))["primary_top10"]
print(f"\nheadline: baseline ROC {m['logistic_baseline']['roc_auc']:.3f}, "
      f"PR-AUC {m['logistic_baseline']['pr_auc']:.3f}; "
      f"GBM ROC {m['gradient_boosting']['roc_auc']:.3f} — baseline wins, as reported.")